# Credit Card Fraud Detection
**Logistic Regression + SVM + Isolation Forest**

Designed for Google Colab. Upload a CSV/Excel dataset containing a fraud/target column (e.g. `Class`, `Fraud`, `IsFraud`).

## 1. Install Required Libraries

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn openpyxl

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import IsolationForest

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import warnings
warnings.filterwarnings("ignore")

print("=" * 70)
print("CREDIT CARD FRAUD DETECTION")
print("Logistic Regression + SVM + Isolation Forest")
print("=" * 70)

print("\nLibraries imported successfully!")

## 3. Upload Dataset

In [ ]:
print("=" * 70)
print("UPLOAD DATASET")
print("=" * 70)

print("\nPlease upload a CSV or Excel dataset.")

uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded.")

file_name = list(uploaded.keys())[0]

print("\nUploaded file:", file_name)

## 4. Load Dataset

In [ ]:
print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

if file_name.lower().endswith(".csv"):

    df = pd.read_csv(file_name)

elif file_name.lower().endswith((".xlsx", ".xls")):

    df = pd.read_excel(file_name)

else:

    raise ValueError(
        "Unsupported file format. "
        "Please upload CSV or Excel."
    )

print("\nDataset loaded successfully!")

print("Rows    :", df.shape[0])
print("Columns :", df.shape[1])

print("\nFirst 5 records:")
display(df.head())

## 5. Dataset Information

In [ ]:
print("=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

total_transactions = len(df)
total_columns = len(df.columns)
missing_values = int(df.isnull().sum().sum())
duplicate_rows = int(df.duplicated().sum())

print("\nTotal Transactions :", total_transactions)
print("Total Columns      :", total_columns)
print("Missing Values     :", missing_values)
print("Duplicate Rows     :", duplicate_rows)

print("\nColumn Names:")

for column in df.columns:
    print(" -", column)

print("\nData Types:")

display(df.dtypes)

## 6. Automatic Target/Fraud Column Detection

In [ ]:
print("=" * 70)
print("TARGET COLUMN DETECTION")
print("=" * 70)

possible_target_columns = [

    "Class",
    "class",

    "Fraud",
    "fraud",

    "IsFraud",
    "is_fraud",

    "Fraud_Flag",
    "fraud_flag",

    "FraudFlag",
    "fraudflag",

    "Target",
    "target",

    "Label",
    "label",

    "Is_Fraud",
    "isFraud"

]

target_column = None

for column in df.columns:

    if column in possible_target_columns:

        target_column = column

        break


if target_column is None:

    print("\nWARNING:")
    print("No common fraud/target column was detected.")

    print("\nAvailable columns:")

    print(list(df.columns))

    print(
        "\nThe supervised models require a fraud/target "
        "column containing legitimate and fraud labels."
    )

    print(
        "\nThe program will stop here because Logistic "
        "Regression and SVM cannot be properly trained "
        "without known target labels."
    )

    raise ValueError(
        "No fraud/target column detected."
    )


print("\nDetected target column:", target_column)

## 7. Check Target Values

In [ ]:
print("=" * 70)
print("TARGET VALUE ANALYSIS")
print("=" * 70)

print("\nTarget value distribution:")

display(
    df[target_column].value_counts()
)

## 8. Convert Target Values to 0/1

In [ ]:
print("=" * 70)
print("TARGET NORMALIZATION")
print("=" * 70)


unique_values = list(
    df[target_column].dropna().unique()
)

print("\nOriginal target values:")

print(unique_values)


# If target is already numeric 0/1
numeric_target = pd.to_numeric(
    df[target_column],
    errors="coerce"
)


if numeric_target.notna().all():

    numeric_unique = sorted(
        numeric_target.unique()
    )

    if set(numeric_unique).issubset({0, 1}):

        df[target_column] = numeric_target.astype(int)

    else:

        # Convert binary numeric target into 0/1
        if len(numeric_unique) == 2:

            mapping = {
                numeric_unique[0]: 0,
                numeric_unique[1]: 1
            }

            df[target_column] = numeric_target.map(
                mapping
            ).astype(int)

        else:

            raise ValueError(
                "Target column must contain two classes."
            )

else:

    # Handle common text labels
    target_text = (
        df[target_column]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    legitimate_words = {
        "0",
        "legitimate",
        "normal",
        "genuine",
        "valid",
        "not fraud",
        "non-fraud",
        "non_fraud"
    }

    fraud_words = {
        "1",
        "fraud",
        "fraudulent",
        "anomaly",
        "anomalous"
    }

    converted_target = []

    for value in target_text:

        if value in legitimate_words:

            converted_target.append(0)

        elif value in fraud_words:

            converted_target.append(1)

        else:

            converted_target.append(np.nan)

    df[target_column] = converted_target

    if df[target_column].isnull().any():

        raise ValueError(
            "Could not understand all target values. "
            "Use 0/1 or labels such as Fraud/Legitimate."
        )

    df[target_column] = (
        df[target_column]
        .astype(int)
    )


# Check that there are exactly two classes

target_classes = sorted(
    df[target_column].unique()
)

if len(target_classes) != 2:

    raise ValueError(
        "The target column must contain exactly two classes."
    )


print("\nNormalized target values:")

print(
    df[target_column].value_counts()
)

## 9. Fraud vs Legitimate Transactions

In [ ]:
print("=" * 70)
print("FRAUD VS LEGITIMATE TRANSACTIONS")
print("=" * 70)

legitimate_count = int(
    (df[target_column] == 0).sum()
)

fraud_count = int(
    (df[target_column] == 1).sum()
)

legitimate_percentage = (
    legitimate_count /
    total_transactions
) * 100

fraud_percentage = (
    fraud_count /
    total_transactions
) * 100

print("\nTotal Transactions :", total_transactions)

print(
    "Legitimate         :",
    legitimate_count,
    f"({legitimate_percentage:.2f}%)"
)

print(
    "Fraudulent         :",
    fraud_count,
    f"({fraud_percentage:.2f}%)"
)

## 10. Visualize Class Distribution

In [ ]:
plt.figure(figsize=(8, 5))

class_counts = [
    legitimate_count,
    fraud_count
]

plt.bar(
    ["Legitimate", "Fraud"],
    class_counts
)

plt.title(
    "Legitimate vs Fraudulent Transactions"
)

plt.xlabel("Transaction Type")

plt.ylabel("Number of Transactions")

plt.show()

## 11. Create Feature Data

In [ ]:
print("=" * 70)
print("PREPARING FEATURES")
print("=" * 70)

data = df.copy()

X = data.drop(
    columns=[target_column]
)

y = data[target_column]

## 12. Remove ID-like Columns

In [ ]:
id_columns = []

for column in X.columns:

    column_lower = column.lower()

    if (
        "transaction_id" in column_lower
        or column_lower == "id"
        or column_lower.endswith("_id")
    ):

        id_columns.append(column)


if id_columns:

    print(
        "\nRemoving ID columns:",
        id_columns
    )

    X = X.drop(
        columns=id_columns
    )

else:

    print(
        "\nNo ID columns detected."
    )

## 13. Handle Date/Time Columns

In [ ]:
for column in X.columns:

    if (
        "date" in column.lower()
        or "time" in column.lower()
    ):

        try:

            converted = pd.to_datetime(
                X[column],
                errors="coerce"
            )

            if converted.notna().mean() > 0.8:

                X[column + "_year"] = (
                    converted.dt.year
                )

                X[column + "_month"] = (
                    converted.dt.month
                )

                X[column + "_day"] = (
                    converted.dt.day
                )

                X[column + "_hour"] = (
                    converted.dt.hour
                )

                X = X.drop(
                    columns=[column]
                )

                print(
                    "Processed date/time column:",
                    column
                )

        except:

            pass

## 14. Handle Categorical Features

In [ ]:
print("\nHandling categorical features...")

categorical_columns = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

if categorical_columns:

    print(
        "Categorical columns:",
        categorical_columns
    )

    X = pd.get_dummies(
        X,
        columns=categorical_columns,
        drop_first=True
    )

else:

    print(
        "No categorical columns detected."
    )

## 15–17. Numeric Conversion, Infinite Values and Missing Values

In [ ]:
# Convert everything to numeric
X = X.apply(
    pd.to_numeric,
    errors="coerce"
)

# Handle infinite values
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

# Handle missing values
print("\nHandling missing values...")

missing_before = int(
    X.isnull().sum().sum()
)

print(
    "Missing values before:",
    missing_before
)

imputer = SimpleImputer(
    strategy="median"
)

X_imputed = imputer.fit_transform(
    X
)

print(
    "Missing values handled successfully."
)

## 18. Feature Scaling

In [ ]:
print("\nScaling features...")

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X_imputed
)

print(
    "Feature scaling completed."
)

## 19. Train / Test Split

In [ ]:
print("=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(

    X_scaled,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print(
    "\nTraining samples:",
    len(X_train)
)

print(
    "Testing samples :",
    len(X_test)
)

## 20. Logistic Regression

In [ ]:
print("=" * 70)
print("LOGISTIC REGRESSION")
print("=" * 70)

logistic_model = LogisticRegression(

    max_iter=1000,

    class_weight="balanced",

    random_state=42

)

logistic_model.fit(
    X_train,
    y_train
)

logistic_pred = logistic_model.predict(
    X_test
)

print(
    "\nLogistic Regression training completed."
)

## 21. Support Vector Machine (SVM)

In [ ]:
print("=" * 70)
print("SUPPORT VECTOR MACHINE")
print("=" * 70)

svm_model = SVC(

    class_weight="balanced",

    random_state=42

)

svm_model.fit(
    X_train,
    y_train
)

svm_pred = svm_model.predict(
    X_test
)

print(
    "\nSVM training completed."
)

## 22. Isolation Forest

In [ ]:
print("=" * 70)
print("ISOLATION FOREST")
print("=" * 70)

# Estimate contamination from the training data.
# A maximum of 10% is used to avoid an extreme setting.
estimated_contamination = (
    y_train.mean()
)

estimated_contamination = float(
    np.clip(
        estimated_contamination,
        0.01,
        0.10
    )
)

print(
    "\nEstimated contamination:",
    estimated_contamination
)

isolation_model = IsolationForest(

    contamination=estimated_contamination,

    random_state=42,

    n_estimators=200

)

isolation_model.fit(
    X_train
)

isolation_raw = isolation_model.predict(
    X_test
)


# Isolation Forest:
#  1  = Normal
# -1  = Anomaly

# Convert:
# Normal  -> 0
# Anomaly -> 1

isolation_pred = np.where(
    isolation_raw == -1,
    1,
    0
)

print(
    "\nIsolation Forest training completed."
)

## 23. Model Evaluation Function

In [ ]:
def evaluate_model(
    model_name,
    y_true,
    y_pred
):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    fraud_detected = int(
        (y_pred == 1).sum()
    )

    return {

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "Fraud Detected": fraud_detected

    }

## 24–25. Evaluate and Compare All Three Models

In [ ]:
results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test,
        logistic_pred
    )
)

results.append(
    evaluate_model(
        "SVM",
        y_test,
        svm_pred
    )
)

results.append(
    evaluate_model(
        "Isolation Forest",
        y_test,
        isolation_pred
    )
)

results_df = pd.DataFrame(
    results
)

print("=" * 70)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 70)

display(
    results_df.style.format({

        "Accuracy": "{:.4f}",

        "Precision": "{:.4f}",

        "Recall": "{:.4f}",

        "F1 Score": "{:.4f}"

    })
)

## 26. Confusion Matrices

In [ ]:
models_predictions = {

    "Logistic Regression":
        logistic_pred,

    "SVM":
        svm_pred,

    "Isolation Forest":
        isolation_pred

}


for model_name, predictions in models_predictions.items():

    cm = confusion_matrix(
        y_test,
        predictions
    )

    plt.figure(
        figsize=(6, 5)
    )

    sns.heatmap(

        cm,

        annot=True,

        fmt="d",

        cmap="Blues",

        xticklabels=[
            "Legitimate",
            "Fraud"
        ],

        yticklabels=[
            "Legitimate",
            "Fraud"
        ]

    )

    plt.title(
        model_name +
        " - Confusion Matrix"
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "Actual"
    )

    plt.show()

## 27. Classification Reports

In [ ]:
for model_name, predictions in models_predictions.items():

    print("\n")
    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print(

        classification_report(

            y_test,

            predictions,

            target_names=[
                "Legitimate",
                "Fraud"
            ],

            zero_division=0

        )

    )

## 28. Fraud Detection Summary

In [ ]:
print("=" * 70)
print("FRAUD DETECTION SUMMARY")
print("=" * 70)

actual_test_fraud = int(
    (y_test == 1).sum()
)

actual_test_legitimate = int(
    (y_test == 0).sum()
)

logistic_fraud = int(
    (logistic_pred == 1).sum()
)

svm_fraud = int(
    (svm_pred == 1).sum()
)

isolation_fraud = int(
    (isolation_pred == 1).sum()
)

print(
    "\nActual fraud in test data:",
    actual_test_fraud
)

print(
    "Actual legitimate in test data:",
    actual_test_legitimate
)

print(
    "\nLogistic Regression frauds detected:",
    logistic_fraud
)

print(
    "SVM frauds detected:",
    svm_fraud
)

print(
    "Isolation Forest anomalies detected:",
    isolation_fraud
)

## 29–30. Prediction Table and Three-Model Voting

In [ ]:
test_indices = y_test.index

prediction_results = df.loc[
    test_indices
].copy()

prediction_results[
    "Logistic_Regression"
] = logistic_pred

prediction_results[
    "SVM"
] = svm_pred

prediction_results[
    "Isolation_Forest"
] = isolation_pred


# Three-model voting
prediction_results[
    "Fraud_Votes"
] = (

    prediction_results[
        "Logistic_Regression"
    ]

    +

    prediction_results[
        "SVM"
    ]

    +

    prediction_results[
        "Isolation_Forest"
    ]

)


def determine_risk(votes):

    if votes >= 2:

        return "HIGH FRAUD RISK"

    elif votes == 1:

        return "POSSIBLE FRAUD"

    else:

        return "LEGITIMATE"


prediction_results[
    "Final_Risk"
] = prediction_results[
    "Fraud_Votes"
].apply(
    determine_risk
)

## 31. Prediction Results

In [ ]:
print("=" * 70)
print("PREDICTION RESULTS")
print("=" * 70)

display(
    prediction_results.head(30)
)

## 32. Fraudulent Transactions Detected

In [ ]:
print("=" * 70)
print("FRAUDULENT TRANSACTIONS DETECTED")
print("=" * 70)

fraud_transactions = prediction_results[
    prediction_results[
        "Final_Risk"
    ].isin([
        "HIGH FRAUD RISK",
        "POSSIBLE FRAUD"
    ])
]

print(
    "\nNumber of suspicious transactions:",
    len(fraud_transactions)
)

display(
    fraud_transactions
)

## 33. High-Risk Transactions

In [ ]:
print("=" * 70)
print("HIGH FRAUD RISK TRANSACTIONS")
print("=" * 70)

high_risk_transactions = prediction_results[
    prediction_results[
        "Final_Risk"
    ] == "HIGH FRAUD RISK"
]

print(
    "\nHigh-risk transaction count:",
    len(high_risk_transactions)
)

display(
    high_risk_transactions
)

## 34. Final Risk Distribution

In [ ]:
print("=" * 70)
print("FINAL RISK DISTRIBUTION")
print("=" * 70)

risk_counts = (
    prediction_results[
        "Final_Risk"
    ]
    .value_counts()
)

display(
    risk_counts
)

## 35. Risk Distribution Graph

In [ ]:
plt.figure(
    figsize=(8, 5)
)

risk_counts.plot(
    kind="bar"
)

plt.title(
    "Final Transaction Risk Distribution"
)

plt.xlabel(
    "Risk Category"
)

plt.ylabel(
    "Number of Transactions"
)

plt.xticks(
    rotation=0
)

plt.show()

## 36. Model Fraud Detection Graph

In [ ]:
model_names = [
    "Logistic Regression",
    "SVM",
    "Isolation Forest"
]

fraud_counts = [
    logistic_fraud,
    svm_fraud,
    isolation_fraud
]

plt.figure(
    figsize=(9, 5)
)

plt.bar(
    model_names,
    fraud_counts
)

plt.title(
    "Fraud/Anomaly Detection by Model"
)

plt.xlabel(
    "Machine Learning Model"
)

plt.ylabel(
    "Fraudulent Transactions Detected"
)

plt.xticks(
    rotation=15
)

plt.show()

## 37. Final Project Summary

In [ ]:
print("=" * 70)
print("CREDIT CARD FRAUD DETECTION - FINAL SUMMARY")
print("=" * 70)

print(
    "\nTotal dataset transactions:",
    len(df)
)

print(
    "Total dataset columns:",
    len(df.columns)
)

print(
    "Total legitimate transactions:",
    legitimate_count
)

print(
    "Total fraudulent transactions:",
    fraud_count
)

print(
    "\nTest dataset transactions:",
    len(y_test)
)

print(
    "Actual fraud in test dataset:",
    actual_test_fraud
)

print(
    "\nModel Results:"
)

for _, row in results_df.iterrows():

    print("\nModel:", row["Model"])

    print(
        "Accuracy :",
        f"{row['Accuracy']:.4f}"
    )

    print(
        "Precision:",
        f"{row['Precision']:.4f}"
    )

    print(
        "Recall   :",
        f"{row['Recall']:.4f}"
    )

    print(
        "F1 Score :",
        f"{row['F1 Score']:.4f}"
    )

    print(
        "Fraud Detected:",
        int(row["Fraud Detected"])
    )


print("\n")
print("=" * 70)
print("PROJECT EXECUTION COMPLETED")
print("=" * 70)

print(
    "\nThe dataset was processed using:"
)

print("1. Logistic Regression")
print("2. Support Vector Machine (SVM)")
print("3. Isolation Forest")

print(
    "\nFraud detection, model evaluation, "
    "confusion matrices and risk analysis "
    "have been completed."
)